<a href="https://colab.research.google.com/github/ArifahZhafirah/PEMBELAJARAN-MESIN/blob/main/JS11/LAB-01/PEMMES1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lab 1
Klasifikasi Cats vs. Dogs

#Deskripsi
Pada praktikum ini kita akan membuat model klasifikasi CNN sederhana pada kasus citra kucing dan anjing.

#Dataset
Dataset merupakan data citra anjing dan kucing yang telah dibagi menjadi data training dan data testing. Dikarenakan data cukup besar, pastikan koneksi Anda sebelum mengunduh dataset.

#Langkah 0 - Menyambungkan Colab ke Google Drive


In [1]:
# Pakai Keras versi lama supaya ImageDataGenerator tersedia
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
!ls dataset

single_prediction  test_set  training_set


In [3]:
!pip install -q gdown
!gdown 1vYrqBI1VmiXXJd5sgtKK2nuQvC8T1ryb -O dataset_catdog.zip
!unzip -q dataset_catdog.zip

Downloading...
From (original): https://drive.google.com/uc?id=1vYrqBI1VmiXXJd5sgtKK2nuQvC8T1ryb
From (redirected): https://drive.google.com/uc?id=1vYrqBI1VmiXXJd5sgtKK2nuQvC8T1ryb&confirm=t&uuid=47d00cbd-2ceb-4ef9-b907-e0d8e66d11ff
To: /content/dataset_catdog.zip
100% 233M/233M [00:01<00:00, 125MB/s]
replace dataset/single_prediction/cat_or_dog_1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


TF_USE_LEGACY_KERAS membuat TensorFlow memakai Keras lama supaya ImageDataGenerator tersedia. gdown dan unzip mengunduh dan mengekstrak dataset ke /content.

#Langkah 1 - Import Library

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

#Langkah 2 - Pra Pengolahan Data
Pada tahap ini kita akan sedikit melakukan manipulasi pada citra yang digunakan. Manipulasi yang dilakukan diantaranya adalah normalisasi nilai piksel, koreksi kemiringan, pembesaran (zoom), dan flip.

#Langkah 2.1. Pra Pengolahan Data Training

In [7]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

Found 8000 images belonging to 2 classes.


#Langkah 2.2. Pra Pengolahan Data Testing

In [8]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


#Langkah 3 - Pembuatan Model CNN
#Langkah 3.1.  - Inisiasi Model CNN

In [9]:
cnn = tf.keras.models.Sequential()

#Langkah 3.2. - Pembuatan Layer Konvolusi 1

In [10]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu',
                               input_shape=[64, 64, 3]))

#Langkah 3.3 - Pembuatan Layer Pooling 1

In [11]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

#Langkah 3.4 - Pembuatan Layer Konvolusi 2 dan Pooling 2

In [12]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

#Langkah 3.5 - Flattening

In [13]:
cnn.add(tf.keras.layers.Flatten())

#Langkah 3.6 - Fully Connected Layer 1 (Input)

In [14]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

#Langkah 3.7 - Fully Connected Layer 2 (Output)

In [15]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

#Langkah 3.8 - Compile Model CNN

In [16]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics =
 ['accuracy'])

Penggunaan loss function binary crossentropy dikarenakan kita hanya melakukan klasifikasi pada dua kelas, yaitu kucing dan anjing.

#Langkah 4 - Fit CNN

In [17]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

Epoch 1/25
250/250 [==============================] - 88s 348ms/step - loss: 0.6469 - accuracy: 0.6151 - val_loss: 0.6199 - val_accuracy: 0.6585
Epoch 2/25
250/250 [==============================] - 81s 323ms/step - loss: 0.5804 - accuracy: 0.6974 - val_loss: 0.5431 - val_accuracy: 0.7350
Epoch 3/25
250/250 [==============================] - 80s 319ms/step - loss: 0.5345 - accuracy: 0.7290 - val_loss: 0.5154 - val_accuracy: 0.7565
Epoch 4/25
250/250 [==============================] - 80s 320ms/step - loss: 0.5117 - accuracy: 0.7449 - val_loss: 0.5294 - val_accuracy: 0.7420
Epoch 5/25
250/250 [==============================] - 80s 319ms/step - loss: 0.4875 - accuracy: 0.7589 - val_loss: 0.5069 - val_accuracy: 0.7575
Epoch 6/25
250/250 [==============================] - 91s 364ms/step - loss: 0.4639 - accuracy: 0.7790 - val_loss: 0.5327 - val_accuracy: 0.7550
Epoch 7/25
250/250 [==============================] - 81s 323ms/step - loss: 0.4475 - accuracy: 0.7912 - val_loss: 0.4709 - val_ac

#Langkah 5 - Prediksi dengan 1 Citra
Pada langkah ini, kita akan mencoba melakukan prediksi pada 1 citra anjing dan kucing.

In [18]:
import numpy as np
from keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

1/1 [==============================] - 0s 142ms/step


In [19]:
import os
for root, dirs, files in os.walk('dataset/training_set'):
    print(root, '| subfolder:', dirs[:5], '| jumlah file:', len(files))
    if root.count('/') >= 4:
        break

dataset/training_set | subfolder: ['cats', 'dogs'] | jumlah file: 1
dataset/training_set/cats | subfolder: [] | jumlah file: 4001
dataset/training_set/dogs | subfolder: [] | jumlah file: 4001


In [20]:
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
print("TF", tf.__version__, "| tf.keras name:", getattr(tf.keras, "__name__", None))
import tf_keras
print("tf_keras", tf_keras.__version__, "| name:", tf_keras.__name__, "| file:", getattr(tf_keras, "__file__", None))
print("pre:", tf_keras.preprocessing.__name__, "| img:", tf_keras.preprocessing.image.__name__)
print("ImageDataGenerator di tf_keras:", hasattr(tf_keras.preprocessing.image, "ImageDataGenerator"))
print("keras*:", sorted(m for m in sys.modules if m == "keras" or m.startswith("keras."))[:8])
print("tf_keras*:", sorted(m for m in sys.modules if m.startswith("tf_keras"))[:8])
k = sys.modules.get("keras")
print("keras top-level di sys.modules:", None if k is None else (k.__name__, getattr(k, "__file__", None)))
print("SELESAI-INSPEKSI2")

TF 2.21.0 | tf.keras name: tensorflow.keras
tf_keras 2.21.0 | name: tf_keras | file: /usr/local/lib/python3.13/dist-packages/tf_keras/__init__.py
pre: tf_keras.preprocessing | img: tf_keras.preprocessing.image
ImageDataGenerator di tf_keras: True
keras*: ['keras', 'keras._tf_keras', 'keras._tf_keras.keras', 'keras._tf_keras.keras.backend', 'keras._tf_keras.keras.layers', 'keras._tf_keras.keras.losses', 'keras._tf_keras.keras.metrics', 'keras._tf_keras.keras.preprocessing']
tf_keras*: ['tf_keras', 'tf_keras.__internal__', 'tf_keras.__internal__.backend', 'tf_keras.__internal__.layers', 'tf_keras.__internal__.losses', 'tf_keras.__internal__.models', 'tf_keras.__internal__.optimizers', 'tf_keras.__internal__.utils']
keras top-level di sys.modules: ('keras', '/usr/local/lib/python3.13/dist-packages/keras/__init__.py')
SELESAI-INSPEKSI2
